<a href="https://colab.research.google.com/github/Faisalt28/Analisis-Sentimen-Aplikasi-Duolingo/blob/main/sentiment_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Library

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Load Dataset

In [ ]:
df = pd.read_csv("duolingo_reviews.csv")
df.head()

,userName,score,content,at
0,Uma Pradhan,5,good,2026-02-20 07:29:13
1,علیرضا فلاحی,5,Nice,2026-02-20 07:28:23
2,Sheela Josh,4,repeated practice... I like this way,2026-02-20 07:28:05
3,SH HS,3,অনেক কিছু শিখতে চেষ্টা করছি,2026-02-20 07:27:17
4,ملینا مهدوی نیا,5,it's really helpful and good!,2026-02-20 07:27:01


# Buang Rating 3 + Label Binary

In [ ]:
df_binary = df[df['score'] != 3].copy()

def binary_label(score):
    if score <= 2:
        return 0  # negatif
    else:
        return 1  # positif

df_binary['label'] = df_binary['score'].apply(binary_label)

print(df_binary['label'].value_counts())


label
1    25729
0     2825
Name: count, dtype: int64


# Balance Dataset

In [ ]:
df_neg = df_binary[df_binary['label'] == 0]
df_pos = df_binary[df_binary['label'] == 1]

min_class = min(len(df_neg), len(df_pos))

df_neg_sample = df_neg.sample(min_class, random_state=42)
df_pos_sample = df_pos.sample(min_class, random_state=42)

df_binary_balanced = pd.concat([df_neg_sample, df_pos_sample])
df_binary_balanced = df_binary_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_binary_balanced['label'].value_counts())


label
1    2825
0    2825
Name: count, dtype: int64


# Cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_binary_balanced['clean_text'] = df_binary_balanced['content'].apply(clean_text)


# Split Data

In [ ]:
X = df_binary_balanced['clean_text']
y = df_binary_balanced['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))


Train: 4520
Test: 1130


# SVM (Baseline ML)

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

y_pred = svm_model.predict(X_test_tfidf)

print("Accuracy Binary SVM:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy Binary SVM: 0.8557522123893805
              precision    recall  f1-score   support

           0       0.88      0.82      0.85       565
           1       0.83      0.89      0.86       565

    accuracy                           0.86      1130
   macro avg       0.86      0.86      0.86      1130
weighted avg       0.86      0.86      0.86      1130



# Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

print("Accuracy LR:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Accuracy LR: 0.8530973451327434
              precision    recall  f1-score   support

           0       0.87      0.83      0.85       565
           1       0.84      0.88      0.86       565

    accuracy                           0.85      1130
   macro avg       0.85      0.85      0.85      1130
weighted avg       0.85      0.85      0.85      1130



# Deep Learning (LSTM)

In [ ]:
max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)


In [ ]:
model = Sequential()
model.add(Embedding(max_words, 128, input_length=max_len))
model.add(SpatialDropout1D(0.3))
model.add(LSTM(128, dropout=0.3, recurrent_dropout=0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)


Epoch 1/5
142/142 ━━━━━━━━━━━━━━━━━━━━ 62s 344ms/step - accuracy: 0.7040 - loss: 0.5933 - val_accuracy: 0.8319 - val_loss: 0.4044
Epoch 2/5
142/142 ━━━━━━━━━━━━━━━━━━━━ 50s 351ms/step - accuracy: 0.8613 - loss: 0.3583 - val_accuracy: 0.8717 - val_loss: 0.3413
Epoch 3/5
142/142 ━━━━━━━━━━━━━━━━━━━━ 80s 340ms/step - accuracy: 0.9109 - loss: 0.2612 - val_accuracy: 0.8735 - val_loss: 0.3449
Epoch 4/5
142/142 ━━━━━━━━━━━━━━━━━━━━ 48s 341ms/step - accuracy: 0.9195 - loss: 0.2392 - val_accuracy: 0.8717 - val_loss: 0.3731
Epoch 5/5
142/142 ━━━━━━━━━━━━━━━━━━━━ 48s 341ms/step - accuracy: 0.9387 - loss: 0.1859 - val_accuracy: 0.8628 - val_loss: 0.3705


In [ ]:
def predict_with_score(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(pad)[0][0]

    label = "positif" if pred > 0.5 else "negatif"
    return label, float(pred)

test_sentences = [
    "This app is amazing and very helpful!",
    "Absolutely love this app, it makes learning fun and easy.",
    "Great features and smooth experience so far.",
    "It works fine but sometimes it lags.",
    "Not bad, but it could be better.",
    "This app is terrible and keeps crashing.",
    "Very disappointed, I lost all my progress.",
    "Worst update ever, full of bugs.",
    "I really wanted to like this app, but it doesn't work properly.",
    "Customer support never responds and the app freezes constantly."
]

for sentence in test_sentences:
    label, score = predict_with_score(sentence)
    print(f"{sentence}")
    print(f"Prediksi: {label} | Confidence: {score:.4f}")
    print("-"*60)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step
This app is amazing and very helpful!
Prediksi: positif | Confidence: 0.9795
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Absolutely love this app, it makes learning fun and easy.
Prediksi: positif | Confidence: 0.9955
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Great features and smooth experience so far.
Prediksi: positif | Confidence: 0.9653
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
It works fine but sometimes it lags.
Prediksi: positif | Confidence: 0.6530
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Not bad, but it could be better.
Prediksi: negatif | Confidence: 0.0825
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
This app is terrible and keeps crashing.
Prediksi: negatif 